# EVE Frontier Spatial Data Explorer

Interactive notebook for exploring spatial data from EVE Frontier static datasets.

This notebook uses modular utilities for data loading, analysis, and visualization.

## Setup and Configuration

Import required libraries and configure the environment.

In [ ]:
# Standard library imports
import os
import sys
from pathlib import Path

# Add project root to path for imports
project_root = Path('/workspace')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Third-party imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# Project imports
from evefrontier_datasets.data_loader import load_eve_data, get_release_info
from evefrontier_datasets.data_analysis import (
    get_database_overview,
    scan_coordinate_tables,
    print_database_summary,
    search_solar_systems,
    find_system_with_most_planets,
    get_solar_system_info,
)
from evefrontier_datasets.visualization import (
    visualize_solar_system,
    plot_2d_scatter,
    plot_3d_scatter,
    explore_coordinates,
)

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("✓ All imports successful")
print(f"✓ Project root: {project_root}")

## Load EVE Frontier Data

Load the latest static data release from the EVE Frontier datasets repository.

The data is automatically downloaded if not present locally.

In [ ]:
# Load data from the latest release
# Set force_download=True to re-download even if file exists
db_data = load_eve_data(release_tag='e6c3', force_download=False)

print(f"\n✅ Data loaded successfully!")
print(f"   Tables available: {len(db_data)}")

## Database Overview

Get a comprehensive summary of the database structure and contents.

In [ ]:
# Print comprehensive database summary
print_database_summary(db_data)

## Explore Spatial Data

Identify and explore tables containing coordinate data.

In [ ]:
# Scan for tables with coordinate data
coordinate_tables = scan_coordinate_tables(db_data)

print("\n📍 Coordinate Tables Found:")
print("=" * 80)

for table_name, coords in coordinate_tables.items():
    print(f"\n{table_name}:")
    print(f"   X Column: {coords['x_col']}")
    print(f"   Y Column: {coords['y_col']}")
    if coords['z_col']:
        print(f"   Z Column: {coords['z_col']}")
    print(f"   Dimensions: {'3D' if coords['has_z'] else '2D'}")
    
    # Get row count
    df = db_data[table_name]
    valid_coords = df.dropna(subset=[coords['x_col'], coords['y_col']])
    print(f"   Valid Points: {len(valid_coords):,}")

## Interactive Coordinate Exploration

Explore coordinate data interactively with statistics and available coloring options.

In [ ]:
# Explore each table with coordinates
print("\n🔍 Exploring Coordinate Tables:\n")
print("=" * 80)

explored_tables = {}
for table_name in coordinate_tables.keys():
    coords = coordinate_tables[table_name]
    explored = explore_coordinates(
        db_data, 
        table_name, 
        coords['x_col'], 
        coords['y_col'], 
        coords['z_col']
    )
    if explored:
        explored_tables[table_name] = explored
    print()

## 2D Spatial Visualizations

Generate 2D scatter plots for all tables with coordinate data.

In [ ]:
# Create 2D plots for each coordinate table
print("\n📊 Generating 2D Spatial Visualizations:\n")
print("=" * 80)

for table_name, coords in coordinate_tables.items():
    print(f"\n📍 {table_name}")
    plot_2d_scatter(
        db_data,
        table_name,
        coords['x_col'],
        coords['y_col'],
        size=50
    )
    print()

## 3D Spatial Visualizations

Generate interactive 3D scatter plots for tables with 3D coordinate data.

In [ ]:
# Create 3D plots for tables with Z coordinates
print("\n📊 Generating 3D Spatial Visualizations:\n")
print("=" * 80)

for table_name, coords in coordinate_tables.items():
    if coords['has_z'] and coords['z_col']:
        print(f"\n📍 {table_name} (3D)")
        plot_3d_scatter(
            db_data,
            table_name,
            coords['x_col'],
            coords['y_col'],
            coords['z_col'],
            size=5
        )
        print()
    else:
        print(f"\n⊘ {table_name} - No Z coordinate, skipping 3D plot")

## Solar System Exploration

Explore individual solar systems and their celestial objects.

### List Available Solar Systems

In [ ]:
# Display available solar systems
if 'SolarSystems' in db_data:
    systems_df = db_data['SolarSystems']
    
    print(f"\n🌍 Solar Systems Database")
    print("=" * 80)
    print(f"Total Systems: {len(systems_df):,}\n")
    
    print("Sample Solar Systems:")
    print("-" * 80)
    for idx, (_, row) in enumerate(systems_df.head(20).iterrows(), 1):
        system_id = row['solarSystemId']
        system_name = row['name']
        
        # Get additional info
        info = get_solar_system_info(db_data, system_id)
        if info:
            print(f"{idx:2}. {system_name:30} (ID: {system_id:8}) | "
                  f"P:{info['planet_count']:2} M:{info['moon_count']:3} S:{info['station_count']:2}")
else:
    print("✗ SolarSystems table not found")

### Search for Solar Systems

In [ ]:
# Search for solar systems by name
search_query = 'Brana'  # Change this to search for different systems

print(f"\n🔎 Searching for systems matching: '{search_query}'")
print("=" * 80)

results = search_solar_systems(db_data, search_query)

if len(results) > 0:
    print(f"\n✓ Found {len(results)} matching system(s):\n")
    
    for idx, (_, row) in enumerate(results.iterrows(), 1):
        system_id = row['solarSystemId']
        system_name = row['name']
        
        # Get detailed info
        info = get_solar_system_info(db_data, system_id)
        if info:
            print(f"{idx}. {system_name}")
            print(f"   ID: {system_id}")
            print(f"   Planets: {info['planet_count']}")
            print(f"   Moons: {info['moon_count']}")
            print(f"   Stations: {info['station_count']}")
            print()
else:
    print(f"\n✗ No systems found matching '{search_query}'")

### Visualize a Specific Solar System

In [ ]:
# Visualize a specific solar system
# Change the search query or system_id to explore different systems

if len(results) > 0:
    # Visualize the first matching system
    selected_system = results.iloc[0]
    system_id = selected_system['solarSystemId']
    system_name = selected_system['name']
    
    print(f"\n⭐ Visualizing Solar System: {system_name}")
    print("=" * 80)
    
    system_plot_data = visualize_solar_system(
        db_data,
        system_id,
        system_name,
        size_scale=1.0
    )
    
    if system_plot_data is not None:
        print(f"\n✅ Visualization complete for {system_name}")
else:
    print("\n⚠️  No system selected for visualization")
    print("   Run the search cell above first")

### Find System with Most Planets

In [ ]:
# Find and visualize the system with the most planets
print("\n🌍 Finding System with Most Planets")
print("=" * 80)

best_system = find_system_with_most_planets(db_data)

if best_system:
    print(f"\n✨ System with Most Planets:")
    print(f"   Name: {best_system['name']}")
    print(f"   ID: {best_system['system_id']}")
    print(f"   Planets: {best_system['planet_count']}")
    print(f"   Moons: {best_system['moon_count']}")
    print(f"   Stations: {best_system['station_count']}")
    print(f"   Total Objects: {best_system['planet_count'] + best_system['moon_count'] + best_system['station_count'] + 1}")
    
    print(f"\n📊 Visualizing {best_system['name']}...")
    print("=" * 80)
    
    best_system_plot = visualize_solar_system(
        db_data,
        best_system['system_id'],
        best_system['name'],
        size_scale=1.0
    )
    
    if best_system_plot is not None:
        print(f"\n✅ Visualization complete")
        
        # Display coordinate statistics
        print(f"\n📈 Coordinate Statistics:")
        print(best_system_plot[['type', 'x_centered', 'y_centered', 'z_centered']].describe())
else:
    print("\n✗ Could not find system data")

## Custom Visualizations

Create custom visualizations by selecting specific tables and parameters.

In [ ]:
# Example: Custom visualization
# Modify these parameters to create your own visualizations

# Select a table with coordinates
selected_table = 'Planets'  # Change to any table from coordinate_tables

if selected_table in coordinate_tables:
    coords = coordinate_tables[selected_table]
    
    print(f"\n🎨 Custom Visualization: {selected_table}")
    print("=" * 80)
    
    # 2D plot
    print(f"\n2D Plot ({coords['x_col']} vs {coords['y_col']}):")
    plot_2d_scatter(
        db_data,
        selected_table,
        coords['x_col'],
        coords['y_col'],
        size=60
    )
    
    # 3D plot if available
    if coords['has_z'] and coords['z_col']:
        print(f"\n3D Plot ({coords['x_col']}, {coords['y_col']}, {coords['z_col']}):")
        plot_3d_scatter(
            db_data,
            selected_table,
            coords['x_col'],
            coords['y_col'],
            coords['z_col'],
            size=8
        )
else:
    print(f"\n✗ Table '{selected_table}' not found in coordinate tables")
    print(f"\nAvailable tables:")
    for table in coordinate_tables.keys():
        print(f"   - {table}")

## Data Analysis

Perform custom data analysis on the loaded tables.

In [ ]:
# Example: Analyze planet distribution by solar system
if 'Planets' in db_data:
    planets_df = db_data['Planets']
    
    print("\n🪐 Planet Distribution Analysis")
    print("=" * 80)
    
    # Count planets per system
    planets_per_system = planets_df['solarSystemId'].value_counts()
    
    print(f"\nTotal Solar Systems with Planets: {len(planets_per_system)}")
    print(f"Total Planets: {len(planets_df):,}")
    print(f"Average Planets per System: {planets_per_system.mean():.2f}")
    print(f"Median Planets per System: {planets_per_system.median():.0f}")
    print(f"Max Planets in One System: {planets_per_system.max()}")
    
    # Distribution histogram
    fig, ax = plt.subplots(figsize=(12, 6))
    planets_per_system.hist(bins=range(0, int(planets_per_system.max()) + 2), ax=ax, edgecolor='black')
    ax.set_xlabel('Number of Planets', fontsize=12)
    ax.set_ylabel('Number of Systems', fontsize=12)
    ax.set_title('Distribution of Planets per Solar System', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print("\n✓ Analysis complete")

## Conclusion

This notebook provides a modular approach to exploring EVE Frontier spatial data.

**Key Features:**
- Automated data loading from releases
- Coordinate detection and analysis
- 2D and 3D visualizations
- Solar system exploration
- Interactive data analysis

**Next Steps:**
- Modify search queries to explore different systems
- Create custom visualizations with different parameters
- Add your own analysis cells below
- Export data for further processing

**Modules Used:**
- `evefrontier_datasets.data_loader` - Data downloading and loading
- `evefrontier_datasets.data_analysis` - Analysis utilities
- `evefrontier_datasets.visualization` - Plotting functions